In [5]:
import pandas as pd
from pathlib import Path
from or_llm_agent.utils import get_project_root

In [6]:
dir_path_solutions = Path(f"{get_project_root()}/solutions")

In [8]:
# After the loop finishes:
df = pd.read_csv(f"{dir_path_solutions}/eval/run_eval_final.csv")

In [10]:
df

,problem_id,version_tag,target_answer,successful_solutions,mae
0,BWOR-022,v00,-3.170086,1,7.298556e-14
1,BWOR-063,v00,16.000000,0,1.000000e+01
2,BWOR-049,v00,1490.000000,1,0.000000e+00
3,BWOR-055,v00,1500.000000,1,0.000000e+00
4,BWOR-069,v00,500.000000,1,0.000000e+00
...,...,...,...,...,...
241,BWOR-080,v13,937151.785700,1,1.524371e-11
242,BWOR-025,v13,81.000000,0,1.000000e+01
243,BWOR-035,v13,20.548322,0,1.000000e+01
244,BWOR-019,v13,9.260000,1,0.000000e+00


In [23]:
df["nummerical_answer"] = 1 
df.loc[df["target_answer"].isnull(), "nummerical_answer"] = 0
df.loc[df["target_answer"].isna(), "nummerical_answer"] = 0
df["valid_mae_answer"] = 1 
df.loc[df["mae"]>0.5, "valid_mae_answer"] = 0
df["valid_mae"] = df["mae"].values 
df.loc[df["valid_mae_answer"]==0, "valid_mae"] = 0

In [24]:
df_g = df[["version_tag", "successful_solutions", "nummerical_answer", "valid_mae_answer", "mae", "valid_mae"]].groupby(by="version_tag").sum()
df_g["mae"] = df_g["mae"]/df_g["nummerical_answer"]
df_g["valid_mae"] = df_g["valid_mae"]/df_g["valid_mae_answer"]


In [25]:
df_g

,successful_solutions,nummerical_answer,valid_mae_answer,mae,valid_mae
version_tag,,,,,
v0,62,80,60,2.750288,0.000383
v00,54,80,53,3.625386,0.000583
v13,59,80,57,3.126227,0.001722


In [3]:
@dataclass
class PipelineResult:
    problem_id: str
    difficulty: str
    domain: str
    version_tag: str
    target_answer: float
    markdown_model: str
    python_code: str
    exec_result: ExecutionResult

In [59]:
# Setup
MODEL = LiteLlm(
    model="openai/qwen38_q3",
    api_base="http://localhost:8000/v1",
    api_key="not-needed",
    temperature=0.7
)

In [4]:
dir_path = Path(f"{get_project_root()}/solutions/result")
files = [f for f in dir_path.iterdir() if f.is_file()]

In [5]:
str(files[0])

'/mnt/datafast/Development/or-llm-agent/solutions/result/BWOR-020_v13.json'

In [6]:
problems_versions = set([((str(f).split("/")[-1]).split(".")[0]) for f in files])
list_problems_versions = []
for pv in problems_versions:
    list_problems_versions.append([pv, pv.split("_")[0], pv.split("_")[-1], int(pv.split("_")[-1].replace("v", "")) ])
list_problems_versions

[['BWOR-034_v7', 'BWOR-034', 'v7', 7],
 ['BWOR-079_v18', 'BWOR-079', 'v18', 18],
 ['BWOR-072_v2', 'BWOR-072', 'v2', 2],
 ['BWOR-020_v2', 'BWOR-020', 'v2', 2],
 ['BWOR-069_v8', 'BWOR-069', 'v8', 8],
 ['BWOR-007_v2', 'BWOR-007', 'v2', 2],
 ['BWOR-028_v2', 'BWOR-028', 'v2', 2],
 ['BWOR-037_v18', 'BWOR-037', 'v18', 18],
 ['BWOR-020_v9', 'BWOR-020', 'v9', 9],
 ['BWOR-059_v18', 'BWOR-059', 'v18', 18],
 ['BWOR-024_v11', 'BWOR-024', 'v11', 11],
 ['BWOR-055_v16', 'BWOR-055', 'v16', 16],
 ['BWOR-040_v11', 'BWOR-040', 'v11', 11],
 ['BWOR-065_v5', 'BWOR-065', 'v5', 5],
 ['BWOR-062_v2', 'BWOR-062', 'v2', 2],
 ['BWOR-007_v10', 'BWOR-007', 'v10', 10],
 ['BWOR-043_v14', 'BWOR-043', 'v14', 14],
 ['BWOR-024_v19', 'BWOR-024', 'v19', 19],
 ['BWOR-062_v8', 'BWOR-062', 'v8', 8],
 ['BWOR-009_v14', 'BWOR-009', 'v14', 14],
 ['BWOR-077_v8', 'BWOR-077', 'v8', 8],
 ['BWOR-021_v14', 'BWOR-021', 'v14', 14],
 ['BWOR-067_v3', 'BWOR-067', 'v3', 3],
 ['BWOR-057_v4', 'BWOR-057', 'v4', 4],
 ['BWOR-006_v7', 'BWOR-006', 'v

In [7]:
with open(Path(f"{get_project_root()}/problems/data/test.jsonl"), "r", encoding="utf-8") as f:
    problems = [json.loads(line) for line in f if line.strip()]

problem_dict = {p["id"]: p for p in problems if "id" in p}

In [8]:
problem_dict

{'BWOR-001': {'id': 'BWOR-001',
  'en_question': '\n    A candy factory uses raw materials A, B, and C to produce three different brands of candy: Brand X, Brand Y, and Brand Z. The content proportions of A, B, and C in each brand, the costs of the raw materials, the monthly usage limits for each raw material, as well as the unit processing cost and selling price of each candy brand are shown in Table 1-17.The question is: How many kilograms of each brand of candy should the factory produce each month in order to maximize its profit?Please formulate a linear programming mathematical model for this problem.\n\\begin{table}[h]\n    \\centering\n    \\caption{Raw Materials and Candy Production Data}\n    \\renewcommand{\\arraystretch}{1.2}\n    \\begin{tabular}{c|ccc|c|c}\n        \\toprule\n        Raw Materials & X & Y & Z & Raw Material Cost(dollar/kg) & Monthly Usage Limit(kg) \\\\\n        \\midrule\n        A & $\\geq 60\\%$ & $\\geq 30\\%$ & & 2.00 & 2000 \\\\\n        B & & & & 1.

In [9]:
N_BATCHES = max([f[-1] for f in list_problems_versions])
N_BATCHES

19

In [13]:
output = []
for j in range(N_BATCHES+1):
    print("--------------------")
    version_tagX = f"v{j}"
    solutions = [f[1] for f in list_problems_versions if f[-1]==j]

    batch_results = []

    for s in solutions:
        file_path = Path(f"{get_project_root()}/solutions/result/{s}_{version_tagX}.json")
        
        if not file_path.exists():
            print(f"Warning: File not found - {file_path}")
            break

        with open(file_path, "r", encoding="utf-8") as f:
            exec_result_raw = json.load(f)
        exec_result = ExecutionResult(
            success=exec_result_raw["success"],
            stdout=exec_result_raw["stdout"],
            stderr=exec_result_raw["stderr"],
            returncode=exec_result_raw["returncode"]
        )

        # markdown = Path(f"solutions/markdown/{r.problem_id}_{version_tag}.md").read_text(encoding="utf-8")
        # summary_text += f"Mathematical Optimization Model:\n{markdown}\n"

        # code = Path(f"solutions/code/{r.problem_id}_{version_tag}.py").read_text(encoding="utf-8")
        # summary_text += f"Python Script:\n{code}\n"

        res = PipelineResult(
                problem_id=s,
                difficulty=problem_dict[s].get("difficulty"),
                domain=problem_dict[s].get("domain"),
                version_tag=version_tagX,
                target_answer=problem_dict[s].get("answer"),
                markdown_model="",
                python_code="",
                exec_result=exec_result
            )
        batch_results.append(res)

    all_solutions = 0
    successful_solutions = 0
    solutions = []
    for br in batch_results:
        target_answer = br.target_answer 
        all_solutions += 1
        successful_solutions += 1    
        try:
            extract_optimal_value = br.exec_result.stdout.replace("OPTIMAL_VALUE: ", "")
            extract_optimal_value = extract_optimal_value.replace("\n", " ")
            if len(extract_optimal_value.split(" "))>1:
                # print("multiple values")
                model_answer = float(extract_optimal_value.split(" ")[0])
            else: 
                model_answer = float(extract_optimal_value)
            solutions.append([target_answer, model_answer, abs(target_answer-model_answer)/abs(target_answer)])
        except:
            successful_solutions -= 1
            print(br.problem_id, f"domain:{br.domain},", f"difficulty:{br.difficulty},", f"target: {target_answer}", f"no OPTIMAL VALUE: {br.exec_result.stdout}")
    if successful_solutions > 0:
        target = [f[0] for f in solutions]
        model = [f[1] for f in solutions]
        per = [min(10.0, f[-1]) for f in solutions]
        mae = sum(per)/successful_solutions
    else: 
        mae = 10
    print(j, all_solutions, successful_solutions, mae)
    print(target, model, per)
    output.append([j, all_solutions, successful_solutions, mae])

--------------------
0 5 5 0.050288872162102274
[20.0, -700.0, 420.3957101, 43200.0, -1990500.0] [20.0, -699.9999999999973, 420.3957100591716, 43200.0, -1490000.0] [0.0, 3.897834436169693e-15, 9.711886456037356e-11, 0.0, 0.2514443607133886]
--------------------
1 5 5 0.0
[126.2, 2.336, 2500.0, 27.0, 161000.0] [126.2, 2.336, 2500.0, 27.0, 161000.0] [0.0, 0.0, 0.0, 0.0, 0.0]
--------------------
BWOR-020 domain:transportation, difficulty:Medium, target: 700.0 no OPTIMAL VALUE: NO_FEASIBLE_SOLUTION
BWOR-007 domain:finance, difficulty:Medium, target: 12.256 no OPTIMAL VALUE: NO_FEASIBLE_SOLUTION
2 5 3 6.167905692361981e-17
[1.2, 53.0, 47.0] [1.2000000000000002, 53.0, 47.0] [1.8503717077085943e-16, 0.0, 0.0]
--------------------
BWOR-067 domain:production_planning, difficulty:Medium, target: 36319.5 no OPTIMAL VALUE: 
BWOR-046 domain:transportation, difficulty:Medium, target: 6100.0 no OPTIMAL VALUE: NO_FEASIBLE_SOLUTION
3 5 3 0.0
[17.0, 913.5, 1490.0] [17.0, 913.5, 1490.0] [0.0, 0.0, 0.0]


In [14]:
output

[[0, 5, 5, 0.050288872162102274],
 [1, 5, 5, 0.0],
 [2, 5, 3, 6.167905692361981e-17],
 [3, 5, 3, 0.0],
 [4, 5, 4, 0.09387755102040816],
 [5, 5, 5, 7.795668872339385e-16],
 [6, 5, 5, 9.302333360112332e-05],
 [7, 5, 5, 0.40573617029055314],
 [8, 5, 5, 0.0],
 [9, 5, 3, 3.3333333333333335],
 [10, 4, 3, 0.14652014652014653],
 [11, 5, 4, 4.5474735088646414e-17],
 [12, 4, 4, 0.0],
 [13, 5, 4, 4.0098348519193014e-17],
 [14, 5, 4, 0.0],
 [15, 5, 4, 2.961305752320005],
 [16, 5, 4, 4.0098348519193014e-17],
 [17, 5, 4, 0.0],
 [18, 5, 4, 0.05],
 [19, 5, 4, 3.7072880572880575]]

In [12]:
all_solutions = 0
successful_solutions = 0
solutions = []
for j, br in enumerate(batch_results):
    target_answer = br.target_answer 
    all_solutions += 1
    successful_solutions += 1    
    try:
        model_answer = float(br.exec_result.stdout.replace("OPTIMAL_VALUE: ", ""))
        solutions.append([target_answer, model_answer, abs(target_answer-model_answer)/target_answer ])
    except:
        model_answer = br.exec_result.stdout
        successful_solutions -= 1
        print(f"no OPTIMAL VALUE: model_answer")
    # print(j, target_answer, model_answer)
print(all_solutions, successful_solutions, sum([f[-1] for f in solutions])/successful_solutions)

no OPTIMAL VALUE: model_answer
5 4 12500.957288057289


In [56]:


# # 3. Create Agents for Pipeline
# formulator = LlmAgent(
#     model=MODEL,
#     name="formulator",
#     instruction=load_prompt("step1_formulation.txt")
# )

# coder = LlmAgent(
#     model=MODEL,
#     name="coder",
#     instruction=load_prompt("step3_code.txt"), 
#     tools=[filesystem_mcp]
# )

# coordinator = LlmAgent(
#     model=MODEL,
#     name="coordinator",
#     instruction=load_prompt("step5_coordinator.txt")
# )

# # # 4. Test Run
# # runner = InMemoryRunner(agent=formulator)
# # result = await runner.run_debug("Optimize production for 2 products given resource limits.", verbose=True)

In [57]:

# # 1. Pick a random problem from BWOR
# problem = load_random_problem("problems/data/test.jsonl")

# print(f"Loaded Problem ID: {problem.id}")
# print(f"Domain: {problem.domain} | Type: {problem.problem_type} | Expected Answer: {problem.answer}")
# print("-" * 50)

# # 2. Pass the question string to your Google ADK agent framework
# message = f"""
# Solve the following Operations Research problem:
# id: {problem.id}
# question: {problem.en_question}
# """

# # Run Step 1 (Formulation)
# runner = InMemoryRunner(agent=formulator)
# result1 = await runner.run_debug(message, verbose=True)

## start

In [58]:
# 3. Create Agents for Pipeline
formulator = LlmAgent(
    model=MODEL,
    name="formulator",
    instruction=load_prompt("step1_formulation.txt")
)

math_modeler = LlmAgent(
    model=MODEL,
    name="math_modeler",
    instruction=load_prompt("step2_math.txt")
)

coder = LlmAgent(
    model=MODEL,
    name="coder",
    instruction=load_prompt("step3_code.txt"),
    tools=[filesystem_mcp]
)

coordinator = LlmAgent(
    model=MODEL,
    name="coordinator",
    instruction=load_prompt("step5_coordinator.txt")
)

In [59]:

def get_event_text(events: Any) -> str:
    """Extracts text content from a list of ADK Events."""
    text_parts = []
    for event in events:
        # Check if the event contains content and parts
        if hasattr(event, "content") and event.content and hasattr(event.content, "parts"):
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    text_parts.append(part.text)
    return "\n".join(text_parts)

# events = await InMemoryRunner(agent=formulator).run_debug(prompt_1)

# # Extract text from the last text-bearing event
# res_text = events[-1].content.parts[0].text if events and events[-1].content else ""

In [60]:


# Load problem
problem = load_random_problem("problems/data/test.jsonl")
print(f"Problem ID: {problem.id} | Domain: {problem.domain} | Target: {problem.answer}\n Questions: {problem.en_question}\n")


Problem ID: BWOR-048 | Domain: inventory | Target: 2340.0
 Questions: 
To ensure flight safety, the engines on the aircraft must be compulsorily replaced and overhauled every six months. A maintenance plant estimates that, starting from the next half-year, the engine replacement demand for a certain type of fighter aircraft over the next three years (i.e., each half-year) will be: 100, 70, 80, 120, 150, 140. When replacing an engine, it can either be replaced with a new one, or with a repaired used engine. It is known that the purchase cost of each new engine is 10 thousand yuan, while repairing used engines has two options: Fast repair: 2 thousand yuan per engine, delivered in half a year (i.e., sent in the current period, ready in the next period); Slow repair: 10,000 yuan per engine, but takes one year for delivery (i.e., sent in the current period, ready in the period after next). The plant has just received this engine replacement and repair task. It is also known that this type o

In [61]:
# Step 1: Formulation
print("--- STEP 1: Formulation ---")
prompt_1 = f"""
Analyze and formulate this Operations Research problem:
id: {problem.id}
question: {problem.en_question}
"""
res_1_events = await InMemoryRunner(agent=formulator).run_debug(prompt_1)
res_1_text = get_event_text(res_1_events)

--- STEP 1: Formulation ---
formulator > # Operations Research Formulation: Engine Replacement and Repair Planning

### 1. Decision Variables

Let $t$ denote the half-year period, where $t = 1, 2, 3, 4, 5, 6$.

*   **$x_t$**: The number of new engines purchased in period $t$.
    *   *Note:* New engines are available for immediate use in period $t$.
*   **$f_t$**: The number of used engines sent for **fast repair** in period $t$.
    *   *Note:* Fast repair takes half a year. Engines sent in period $t$ are ready for use in period $t+1$. Therefore, $f_6$ is not useful for meeting demand within the horizon (since the aircraft are decommissioned), but we will define variables for $t=1$ to $5$ for repairs that can be utilized.
*   **$s_t$**: The number of used engines sent for **slow repair** in period $t$.
    *   *Note:* Slow repair takes one year (two half-year periods). Engines sent in period $t$ are ready for use in period $t+2$. Therefore, $s_t$ is useful for $t=1$ to $4$.

*(Note: W

In [62]:
# Step 2: Math Modeling
print("--- STEP 2: Mathematical Model ---")
prompt_2 = f"Problem ID:{problem.id}\nProblem:\n{problem.en_question}\n\nFormulation:\n{res_1_text}"
res_2_events = await InMemoryRunner(agent=math_modeler).run_debug(prompt_2)
res_2_text = get_event_text(res_2_events)


--- STEP 2: Mathematical Model ---
math_modeler > To formulate this engine replacement and repair planning problem, we define the following mathematical model.

### 1. Index Sets and Parameters

**Index Sets:**
*   $T = \{1, 2, 3, 4, 5, 6\}$: The set of half-year periods over the 3-year horizon.

**Parameters:**
*   $D_t$: The number of engines requiring replacement in period $t$.
    *   $D = [100, 70, 80, 120, 150, 140]$
*   $C_N = 10$: The cost (in thousand yuan) to purchase one new engine.
*   $C_F = 2$: The cost (in thousand yuan) for fast repair of one used engine.
*   $C_S = 10$: The cost (in thousand yuan) for slow repair of one used engine.
*   $L_F = 1$: The lead time (in periods) for fast repair.
*   $L_S = 2$: The lead time (in periods) for slow repair.

### 2. Decision Variables

Let the decision variables represent the actions taken in each period $t \in T$:

*   $x_t$: The number of **new engines purchased** in period $t$.
    *   Domain: $x_t \in \mathbb{Z}^+$ (non-nega

In [66]:
from IPython.display import display, Markdown
display(Markdown(res_2_text))

To formulate this engine replacement and repair planning problem, we define the following mathematical model.

### 1. Index Sets and Parameters

**Index Sets:**
*   $T = \{1, 2, 3, 4, 5, 6\}$: The set of half-year periods over the 3-year horizon.

**Parameters:**
*   $D_t$: The number of engines requiring replacement in period $t$.
    *   $D = [100, 70, 80, 120, 150, 140]$
*   $C_N = 10$: The cost (in thousand yuan) to purchase one new engine.
*   $C_F = 2$: The cost (in thousand yuan) for fast repair of one used engine.
*   $C_S = 10$: The cost (in thousand yuan) for slow repair of one used engine.
*   $L_F = 1$: The lead time (in periods) for fast repair.
*   $L_S = 2$: The lead time (in periods) for slow repair.

### 2. Decision Variables

Let the decision variables represent the actions taken in each period $t \in T$:

*   $x_t$: The number of **new engines purchased** in period $t$.
    *   Domain: $x_t \in \mathbb{Z}^+$ (non-negative integers)
*   $f_t$: The number of used engines **sent for fast repair** in period $t$.
    *   Domain: $f_t \in \mathbb{Z}^+$ (non-negative integers)
    *   *Note:* Engines sent in period $t$ become available for installation in period $t+1$.
*   $s_t$: The number of used engines **sent for slow repair** in period $t$.
    *   Domain: $s_t \in \mathbb{Z}^+$ (non-negative integers)
    *   *Note:* Engines sent in period $t$ become available for installation in period $t+2$.

*(Note: For periods where repairs are initiated but the resulting engines would be available after the horizon ends (i.e., $f_6$ and $s_5, s_6$), these variables will naturally be zero in the optimal solution as they incur cost without contributing to meeting the demand within $T$. However, they are formally included in the summation for completeness, or can be restricted to 0. Here, we include them in the constraints for periods $t=1$ to $6$ as defined by the source availability.)*

### 3. Objective Function

Minimize the total maintenance cost $Z$ (in thousand yuan):

$$
\text{Minimize } Z = \sum_{t=1}^{6} (C_N x_t + C_F f_t + C_S s_t)
$$

Substituting the parameter values:

$$
Z = 10 \sum_{t=1}^{6} x_t + 2 \sum_{t=1}^{6} f_t + 10 \sum_{t=1}^{6} s_t
$$

### 4. Operational Constraints

**A. Demand Satisfaction Constraints (Flow Balance)**
The total number of engines available for installation in period $t$ must exactly equal the demand $D_t$. The sources of engines available in period $t$ are:
1.  New engines purchased in period $t$ ($x_t$).
2.  Fast repairs initiated in the previous period $t-1$ ($f_{t-1}$).
3.  Slow repairs initiated in the period $t-2$ ($s_{t-2}$).

Assuming no initial inventory of engines in repair or stock at $t=0$ (i.e., $f_0=0, s_0=0, s_{-1}=0$):

*   **Period 1:**
    $$ x_1 = 100 $$
*   **Period 2:**
    $$ x_2 + f_1 = 70 $$
*   **Period 3:**
    $$ x_3 + f_2 + s_1 = 80 $$
*   **Period 4:**
    $$ x_4 + f_3 + s_2 = 120 $$
*   **Period 5:**
    $$ x_5 + f_4 + s_3 = 150 $$
*   **Period 6:**
    $$ x_6 + f_5 + s_4 = 140 $$

**B. Repair Source Constraints (Availability of Used Engines)**
The number of engines sent for repair (fast or slow) in any period $t$ cannot exceed the number of engines removed from the aircraft in that same period. Since one engine is removed for every replacement, the total engines removed in period $t$ is $D_t$.

*   **Period 1:**
    $$ f_1 + s_1 \le 100 $$
*   **Period 2:**
    $$ f_2 + s_2 \le 70 $$
*   **Period 3:**
    $$ f_3 + s_3 \le 80 $$
*   **Period 4:**
    $$ f_4 + s_4 \le 120 $$
*   **Period 5:**
    $$ f_5 + s_5 \le 150 $$
*   **Period 6:**
    $$ f_6 + s_6 \le 140 $$

**C. Non-negativity and Integrality**
$$ x_t, f_t, s_t \ge 0 \quad \forall t \in T $$
$$ x_t, f_t, s_t \in \mathbb{Z} \quad \forall t \in T $$

In [63]:
# Step 3: Code Generation
prompt_3 = f"Problem ID: {problem.id}\nConvert this math model into OR-Tools Python code:\n{res_2_text}"
res_3_events = await InMemoryRunner(agent=coder).run_debug(prompt_3)
res_3_text = get_event_text(res_3_events)


coder > Let me first check the workspace directory structure to determine where to save the script.


coder > Now I'll create the Python script that translates this engine replacement and repair planning model into OR-Tools code.


coder > Let me verify the file was written correctly:


coder > The script is saved and verified. Here's a summary of the translation:

### File: `/mnt/datafast/Development/or-llm-agent/solutions/engine_replacement_planning.py`

**Key design decisions:**

| Aspect | Choice | Rationale |
|---|---|---|
| **Solver** | `SCIP` (MILP) | All variables are integer (`IntVar`), so a MILP solver is required |
| **Variables** | `x[0..5]`, `f[0..5]`, `s[0..5]` as `IntVar(0, ∞)` | 18 integer decision variables covering all 6 periods |
| **Objective** | Minimize `10·Σx + 2·Σf + 10·Σs` | Direct transcription of the cost formula |
| **Demand constraints** | 6 equality constraints | Each period's available engines (new + fast from t−1 + slow from t−2) must exactly meet demand

In [64]:
# for event in res_3_events:
#     print(event)
#     print("-----------")
print(res_3_text)

Let me first check the workspace directory structure to determine where to save the script.


Now I'll create the Python script that translates this engine replacement and repair planning model into OR-Tools code.


Let me verify the file was written correctly:


The script is saved and verified. Here's a summary of the translation:

### File: `/mnt/datafast/Development/or-llm-agent/solutions/engine_replacement_planning.py`

**Key design decisions:**

| Aspect | Choice | Rationale |
|---|---|---|
| **Solver** | `SCIP` (MILP) | All variables are integer (`IntVar`), so a MILP solver is required |
| **Variables** | `x[0..5]`, `f[0..5]`, `s[0..5]` as `IntVar(0, ∞)` | 18 integer decision variables covering all 6 periods |
| **Objective** | Minimize `10·Σx + 2·Σf + 10·Σs` | Direct transcription of the cost formula |
| **Demand constraints** | 6 equality constraints | Each period's available engines (new + fast from t−1 + slow from t−2) must exactly meet demand |
| **Source constraints** | 6 

In [ ]:
# Step 4: Execution
print("--- STEP 4: Execution ---")
exec_res = run_generated_code(res_3_text)


In [ ]:
# Step 5: Coordinator Debug Loop (if needed)
if not exec_res.success:
    print("⚠️ Code failed. Entering Step 5 Debugger...")
    prompt_5 = f"""
    Code failed execution.
    
    Generated Code:
    {res_3_text}
    
    Error Traceback:
    {exec_res.stderr}
    """
    res_5_events = await InMemoryRunner(agent=coordinator).run_debug(prompt_5, verbose=False)
    print("Coordinator Diagnosis:\n", get_event_text(res_5_events))
else:
    print("✅ Solver Output:\n", exec_res.stdout)

In [25]:
# Step 1: Formulation
prompt_1 = f"Analyze and formulate this OR problem:\n{problem.en_question}"
res_1 = await InMemoryRunner(agent=formulator).run_debug(prompt_1)

# Step 3: Code Generation (directly from Step 1)
prompt_3 = f"Problem:\n{problem.en_question}\n\nFormulation:\n{res_1.output_text}"
res_3 = await InMemoryRunner(agent=coder).run_debug(prompt_3)

# Step 4: Run Code
exec_res = run_generated_code(res_3.output_text)

❌ FAILED (BWOR-017): Expected 300.0, got 91.0
